In [3]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [4]:
from dotenv import load_dotenv
load_dotenv()

True

In [5]:
from openai import OpenAI
openai_client =OpenAI()

In [6]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [7]:
len(documents)

72

In [10]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
len(chunks)

295

In [12]:
##Create a Index and searching for the document

from minsearch import Index

In [13]:
index_search = Index(
    text_fields=['content'],
    keyword_fields=['filename']
)

index_search.fit(chunks)

In [14]:
def search(query):
    return index_search.search(
        query,
        boost_dict={'content':3.0},
        num_results=5
    )

In [67]:
question = search('How does the agentic loop keep calling the model until it stops?')
question

[{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry

In [15]:
def search(query):
    return index_search.search(
        query,
        boost_dict={'content':3.0},
        num_results=5
    )

In [16]:
def build_context(search_results):
    results = []
    for doc in search_results:
        results.append("content: " + doc['content']) 
        results.append("filename: " + doc['filename'])
        results.append("")
    
    return '\n'.join(results).strip()

In [17]:
instructions = """
You're a course teaching assistant. 
Answer the student's question using the search tool. 
Make multiple searches with different keywords before answering.
"""

user_prompt = """
Question : 
{question}

Context:
{context}
"""

In [18]:
def build_prompt(query, chunks):
    context = build_context(chunks)
    prompt = user_prompt.format(
        question = query,
        context = context
    )

    return prompt.strip()

In [19]:
def llm(instructions, user_prompt, model='gpt-5.4-mini'):
    messages = [
        {
            "role": "developer",
            "content": instructions
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]
    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response

In [20]:
def rag(query, model='gpt-5.4-mini'):
    search_results = search(query)
    prompt_results = build_prompt(query, search_results)
    answer = llm(instructions, prompt_results, model=model)

    return answer

In [21]:
ans = rag('How does the agentic loop keep calling the model until it stops?')

In [22]:
ans.output[0].content[0].text

'It keeps calling the model inside a `while True` loop and only breaks when the model stops returning tool calls.\n\nHow it works:\n\n1. Call the model with the current `messages`.\n2. Look through `response.output`.\n3. If there’s a `function_call`, run the tool, append the tool result to `messages`, and set `has_function_calls = True`.\n4. If there’s a normal assistant `message`, store/print it.\n5. After processing the output:\n   - if `has_function_calls` is `True`, loop again\n   - if `has_function_calls` is `False`, `break`\n\nSo the stop condition is simple:\n\n- **No function calls in this turn = final answer = exit the loop**\n\nThis means the model decides when it’s done searching, and your code just keeps feeding the results back until the model returns a response with no more tool calls.\n\nA compact summary:\n\n```python\nwhile True:\n    response = model(...)\n    if response includes tool calls:\n        run tools\n        send tool outputs back\n    else:\n        break

In [ ]:
ans.output_text

'The agentic loop keeps calling the model by using a `while True` loop and checking whether the model asked for any tool calls.\n\nIn each iteration:\n\n1. Send the full `messages` history to the model.\n2. Read `response.output`.\n3. If there is a `function_call`, run the tool and append the tool result to `messages`.\n4. Set `has_function_calls = True`.\n5. If there were no function calls, `break` out of the loop.\n\nSo the stop condition is:\n\n- **no function calls in the latest response**\n\nThat means the model has produced its final answer, so the loop ends.\n\nVery simply:\n\n```python\nwhile True:\n    response = model(...)\n    if response has function calls:\n        run tools\n        continue\n    else:\n        break\n```\n\nThe model decides **how many times** to search; your code just keeps looping until it stops asking for tools.'

In [23]:
ans.usage.input_tokens

2297

In [24]:
ans.usage

ResponseUsage(input_tokens=2297, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=256, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=2553)

In [9]:
len(chunks)

295

In [25]:
!uv add toyaikit

Resolved 129 packages in 5.16s                                       
⠙ Preparing packages... (0/7)                                                   ⠋ Preparing packages... (0/0)                                                   
⠙ Preparing packages... (0/7)-------------------     0 B/21.96 KiB           
⠙ Preparing packages... (0/7)---------- 14.79 KiB/21.96 KiB         
⠙ Preparing packages... (0/7)---------- 14.79 KiB/21.96 KiB         
docstring-parser     ------------------------------ 14.79 KiB/21.96 KiB
⠙ Preparing packages... (0/7)-------------------     0 B/902.20 KiB          
docstring-parser     ------------------------------ 14.79 KiB/21.96 KiB
⠙ Preparing packages... (0/7)------------------- 16.00 KiB/902.20 KiB        
docstring-parser     ------------------------------ 14.79 KiB/21.96 KiB
⠙ Preparing packages... (0/7)------------------- 16.00 KiB/902.20 KiB        
docstring-parser     ------------------------------ 14.79 KiB/21.96 KiB
⠙ Preparing packages... (0/7)--

In [26]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [30]:
def search(query: str) -> dict[str,str]:
    """
        Search the index_search for entries matching the given query
    """
    return index_search.search(
        query,
        boost_dict={'content':3.0},
        num_results=5
    )

In [31]:
agent_tool = Tools()
agent_tool.add_tool(search)

In [32]:
agent_tool.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the index_search for entries matching the given query',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [33]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

In [34]:
runner = OpenAIResponsesRunner(
    tools=agent_tool,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model='gpt-5.4-mini')
)

In [ ]:
result = runner.loop(
    prompt='How does the agentic loop keep calling the model until it stops?',
    callback=callback
)



SyntaxError: invalid syntax. Perhaps you forgot a comma? (932069389.py, line 3)